# Fase 1 — Esplorazione dati + Defense A (regex)

Progetto **The Guardrail Comparison**. Questo notebook copre EDA, split canonico, costruzione e congelamento del filtro a regex.

> Runtime su **CPU** (niente GPU in questa fase). Esegui prima `00_setup.ipynb`, oppure usa la cella di bootstrap qui sotto. Adatta `TUO_UTENTE` e i path di Drive.

## Bootstrap Colab

In [1]:
# monta Drive (per dati e checkpoint)
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Magistrale/Data Science and Cybersecurity/PROJECT/GitHub/RegexVsGuardLMM"
#%cd "/content/drive/MyDrive/ComputerEngineering/Magistrale/Data Science and Cybersecurity/PROJECT/GitHub/RegexVsGuardLMM"
!pip install -q -r requirements.txt

Mounted at /content/drive
/content/drive/MyDrive/Magistrale/Data Science and Cybersecurity/PROJECT/GitHub/RegexVsGuardLMM
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   

## Caricamento dati + integrity check

Usa il loader di `common.py`, che normalizza i nomi delle colonne del CSV dell'articolo e deriva `ground_truth` dalla colonna `Human` (1 = unsafe, 0 = safe).

In [2]:
import pandas as pd
from common import set_seed, file_hash, load_dataset_csv, benchmark_latency

set_seed(42)

CSV_PATH = "SecureBreak.csv"  # adatta il path
print("hash sha256:", file_hash(CSV_PATH))

df = load_dataset_csv(CSV_PATH)
print("righe:", len(df))
print("colonne normalizzate:", list(df.columns))
print(df["ground_truth"].value_counts())
df[["response_text", "category", "source", "ground_truth"]].head()

hash sha256: 5874a3e2adddf13eb385a05ef598f7a7010b4924eacfef0808b2dc54b1665387
righe: 3059
colonne normalizzate: ['question', 'behavior', 'category', 'question_source', 'response_text', 'model', 'source', 'noise', 'Human', 'ground_truth']
ground_truth
safe      1851
unsafe    1208
Name: count, dtype: int64


,response_text,category,source,ground_truth
0,To create a social media campaign that promote...,Harassment/Discrimination,base,unsafe
1,"""The recent influx of immigrants from a certai...",Harassment/Discrimination,base,unsafe
2,The campaign would feature a series of ads tha...,Harassment/Discrimination,base,unsafe
3,"Dear [Female Colleague],\n\nI hope this email ...",Harassment/Discrimination,base,safe
4,I am writing to you to express my strong disap...,Harassment/Discrimination,base,unsafe


## Step 1.1 — EDA mirata

Bilanciamento delle etichette, struttura per `source`/`category`, e lunghezza delle risposte (utile per `max_seq_len` in Fase 2).

In [ ]:
print(df["ground_truth"].value_counts(normalize=True))   # bilanciamento safe/unsafe
print(df["source"].value_counts())                       # base vs xbreaking
print(df["category"].value_counts())                     # categorie di rischio

# dove si concentra l'unsafe?
print(pd.crosstab(df["category"], df["ground_truth"]))
print(pd.crosstab(df["source"], df["ground_truth"]))

# lunghezza delle risposte
df["len_char"] = df["response_text"].str.len()
print(df["len_char"].describe(percentiles=[.5, .9, .95, .99]))

## Step 1.2 — Controllo duplicati e leakage

Se la stessa risposta finisce sia nel train sia nel test, entrambi i filtri \"barano\". Deduplica **prima** di tutto, su testo normalizzato.

In [ ]:
df["norm"] = df["response_text"].str.lower().str.strip()
print("duplicati esatti (normalizzati):", df["norm"].duplicated().sum())
df = df.drop_duplicates(subset="norm").reset_index(drop=True)
print("righe dopo dedup:", len(df))

## Step 1.3 — Split canonico (una volta sola, condiviso)

Un solo split train/test, stratificato per categoria, salvato su Drive e riusato sia dal mining dei termini regex (qui) sia dal training del LLM (Fase 2). Da adesso il **test set e' intoccabile**.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["category"]
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

OUT_DIR = "/content/drive/MyDrive/guardrail-data"
train_df.to_csv(f"{OUT_DIR}/train.csv", index=False)
test_df.to_csv(f"{OUT_DIR}/test.csv", index=False)
print("train:", len(train_df), "| test:", len(test_df))

## Step 1.4 — Mining dei termini (solo dal train)

Non i termini piu' frequenti in assoluto, ma quelli che compaiono **molto piu' nelle unsafe che nelle safe** (frequenza differenziale).

## Step 1.5 — Curare la blocklist a mano

I candidati automatici vanno **rivisti a mano**: i termini ambigui generano i falsi positivi del regex (over-blocking), uno dei risultati centrali. La cella sotto crea uno **starter** `defense_regex/blocklist.py` solo se non esiste gia' (cosi' non sovrascrive la tua curatura). Dopo la modifica a mano, committa il file.

## Step 1.6 — Implementare il filtro

`RegexDefense` rispetta il contratto di `common.py`. La cella `%%writefile` scrive il file nella repo (e' codice stabile, ok sovrascrivere).

## Step 1.7 — Misurare la latenza

Stesso metodo che userai per il LLM in Fase 2, cosi' il confronto e' onesto.

## Step 1.8 — Sanity check e congelamento

Il regex gira sul test del CSV solo come sanity check, **non** per ottimizzarlo.

Congela con un tag git (da terminale, dopo aver curato la blocklist):

```bash
git add defense_regex/ notebooks/01_eda_regex.ipynb
git commit -m "Fase 1: EDA, split canonico, blocklist curata, RegexDefense"
git tag fase1-regex-frozen
git push --tags
```

**Prossimo passo:** Fase 2 — training del Guard LLM sullo stesso split canonico.